In [0]:
%sql
-- # Trying to fetch data from Drive 


SELECT
  COUNT(*)
FROM
  `google_drive`.`stock_prices_stock_prices` /* {"destination": "Databricks", "connector": "google_drive", "groupId": "flatterer_venture"} */


In [0]:
%sql

SELECT
  *
FROM
  `google_drive`.`stock_prices_stock_prices`

In [0]:
dbutils.help()

In [0]:
dbutils.fs.ls("/")

# Reading data from hugging facce
[link](https://huggingface.co/buckets/manish95/HF_public95/resolve/5000%20Sales%20Records.csv?download=true)

In [0]:
%pip install datasets==2.18.0

In [0]:
%pip install huggingface_hub --upgrade --no-deps

In [0]:
dbutils.library.restartPython()

In [0]:
# Workaround: Load IMDB dataset manually to avoid Databricks runtime patch conflict
import pandas as pd
from sklearn.datasets import fetch_openml

# Fetch IMDB dataset from OpenML (alternative source)
try:
    data = fetch_openml('imdb', version=1, as_frame=True, parser='auto')
    dataset = data.frame
    print(f"Dataset loaded: {dataset.shape[0]} rows, {dataset.shape[1]} columns")
    display(dataset.head())
except Exception as e:
    print(f"OpenML method failed: {e}")
    print("\nAlternative: Download IMDB dataset manually from https://ai.stanford.edu/~amaas/data/sentiment/")

In [0]:
import pandas as pd 
url = "https://huggingface.co/buckets/manish95/HF_public95/resolve/5000%20Sales%20Records.csv?download=true"

df = pd.read_csv(url)
df.head()
df.columns
df.describe()
df.info()
df.isnull().sum()



In [0]:
!pip install pyspark

In [0]:
%restart_python

In [0]:
from pyspark.sql import SparkSession 
spark = SparkSession.builder.appName("Spark").getOrCreate()

In [0]:
import pandas as pd
from pyspark.sql import SparkSession

# Download CSV using pandas first (works with HTTP URLs)
url = "https://huggingface.co/buckets/manish95/HF_public95/resolve/5000%20Sales%20Records.csv?download=true"
df_pandas = pd.read_csv(url)

# Convert to Spark DataFrame
spark = SparkSession.builder.appName("Spark").getOrCreate()
df_csv = spark.createDataFrame(df_pandas)

df_csv.show()
# df_csv.printSchema()
# df_csv.columns
# df_csv.describe()
# df_csv.count()
# df_csv.groupBy("Region").count().show()

In [0]:
%sql
SHOW TABLES;

In [0]:
%sql
    CREATE DATABASE IF NOT EXISTS `my_db`;
    
    USE my_db;
    


In [0]:
df_csv.write.format("delta").mode("overwrite").save("/FileStore/tables/manish95/delta_sales")

In [0]:
fil500 = df_csv.filter((df_csv["Unit Price"] > 500))
# fil500 = df_csv.filter((df_csv["Unit Price"] > 500) & (df_csv["Region"] == "Asia"))

display(fil500)

In [0]:
fill600 = df_csv.filter("`Unit Price` > 500 ")
# fill600 = df_csv.filter("`Total Revenue` > 500")
fill600.count()

In [0]:
rows = fil500.count()
cols = len(fil500.columns)
print(f"Shape of fil500: ({rows}, {cols})")

In [0]:
avg_sales_per_region = df_csv.groupBy("Region").avg("Units Sold")
display(avg_sales_per_region)

In [0]:
avg_unit_price = df_csv.agg({"Unit Price": "avg"})
display(avg_unit_price)

In [0]:
# Common methods to handle null values in Spark DataFrames with examples:

examples = [
    ("dropna", "df_csv.dropna()  # Drops rows with any null values"),
    ("fillna", "df_csv.fillna({'Units Sold': 0})  # Replaces nulls in 'Units Sold' with 0"),
    ("replace", "df_csv.replace({None: 0})  # Replaces None (null) with 0"),
    ("isNull/isNotNull", "df_csv.filter(df_csv['Units Sold'].isNull())  # Filters rows where 'Units Sold' is null"),
    ("na.subset", "df_csv.dropna(subset=['Units Sold'])  # Drops rows with nulls in 'Units Sold' only"),
    ("coalesce", "df_csv.selectExpr('coalesce(`Units Sold`, 0) as Units_Sold')  # Returns first non-null value"),
    ("when/otherwise", "from pyspark.sql.functions import when\n"
                       "df_csv.withColumn('Units Sold', when(df_csv['Units Sold'].isNull(), 0).otherwise(df_csv['Units Sold']))  # Conditional replacement")
]

print(f"Number of methods to handle null values: {len(examples)}")
for name, ex in examples:
    print(f"- {name}: {ex}")

In [0]:
from pyspark.sql.functions import col, sum

missing_counts = df_csv.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df_csv.columns
])

missing_counts.show()